# L5 · Lab A — Build a Reward Model from Human Preferences

**RL for LLMs / RLHF — Vizuara AI Labs · Dr. Rajat Dandekar**

In the lecture we said: *there is no formula for a "good" answer, so we **learn** one from human comparisons.* This lab builds that learned reward model from scratch on the real **Anthropic HH-RLHF** preference data.

You will:
1. Load real human preference pairs `(prompt, chosen, rejected)`.
2. Put a **scalar head** on a transformer (gpt2) — the lecture's "swap the language head for a head that outputs one number."
3. Train it with the **Bradley-Terry** loss so it scores the *chosen* answer above the *rejected* one (**your one job**).
4. Watch preference accuracy climb above chance.

> Runtime: **GPU** (Runtime → Change runtime type → T4). ~10 minutes.


In [ ]:
# ✅ PROVIDED — setup
!pip -q install "transformers==4.45.2" "datasets==3.0.1" >/dev/null
import torch, torch.nn.functional as F, random
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1 · The data — real human preferences

Each row of HH-RLHF is two full conversations that share a prompt and differ in the assistant's reply: a **chosen** (human-preferred) one and a **rejected** one.

In [ ]:
# ✅ PROVIDED — load a slice of Anthropic HH-RLHF
ds = load_dataset("Anthropic/hh-rlhf", split="train").shuffle(seed=0)
train = ds.select(range(8000))
test  = ds.select(range(8000, 8800))

ex = train[0]
print("CHOSEN  (preferred):\n", ex["chosen"][-400:])
print("\nREJECTED:\n", ex["rejected"][-400:])

## 2 · The model — a transformer with a scalar head

A reward model is just a transformer whose language head is replaced by a **single linear layer to one number**. In 🤗 Transformers that is exactly `AutoModelForSequenceClassification(..., num_labels=1)`.

**Why left-truncation?** The chosen and rejected texts share the prompt and differ in the assistant's reply *at the end*. If we truncated from the right we'd cut off the very part that differs. So we keep the **end**.

In [ ]:
# ✅ PROVIDED — tokenizer + transformer-with-a-scalar-head
tok = AutoTokenizer.from_pretrained("gpt2")
tok.pad_token = tok.eos_token
tok.truncation_side = "left"          # keep the ENDING, where chosen/rejected differ

model = AutoModelForSequenceClassification.from_pretrained("gpt2", num_labels=1)
model.config.pad_token_id = tok.eos_token_id
model.to(device)

MAX_LEN, BS = 512, 8
def score(texts):
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(device)
    return model(**enc).logits.squeeze(-1)   # one scalar reward per text
print("reward model ready — params:", sum(p.numel() for p in model.parameters())//1_000_000, "M")

## 3 · Evaluation — preference accuracy

A reward model is "good" if it scores the chosen answer above the rejected one. On a random model that happens ~50% of the time.

In [ ]:
# ✅ PROVIDED — accuracy = fraction of test pairs where r(chosen) > r(rejected)
@torch.no_grad()
def eval_acc():
    model.eval(); correct = total = 0
    for i in range(0, len(test), BS):
        rc = score(test["chosen"][i:i+BS]); rr = score(test["rejected"][i:i+BS])
        correct += (rc > rr).sum().item(); total += len(rc)
    model.train(); return correct / total

print("accuracy BEFORE training:", round(eval_acc(), 4))   # ~0.50 (chance)

## 4 · 📝 TODO — the Bradley-Terry loss

This is the heart of the lab. We want **P(chosen ≻ rejected) = σ(r_chosen − r_rejected)** to be large. Maximizing its log-likelihood = minimizing `−log σ(r_chosen − r_rejected)`.

Implement that one line. *(Hint: `F.logsigmoid`.)*

In [ ]:
def bradley_terry_loss(r_chosen, r_rejected):
    # 📝 TODO: return the Bradley-Terry loss for a batch of (chosen, rejected) scores.
    # Want sigmoid(r_chosen - r_rejected) -> 1, i.e. minimize -log sigmoid(margin).
    raise NotImplementedError("implement the Bradley-Terry loss")

In [ ]:
# ✅ PROVIDED — training loop (calls your loss)
opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
idx = list(range(len(train))); random.seed(0); random.shuffle(idx)
acc_curve = [(0, eval_acc())]; step = 0
for i in range(0, len(idx), BS):
    b = idx[i:i+BS]
    rc = score([train["chosen"][j]   for j in b])
    rr = score([train["rejected"][j] for j in b])
    loss = bradley_terry_loss(rc, rr)            # <-- your TODO
    opt.zero_grad(); loss.backward(); opt.step(); step += 1
    if step % 100 == 0:
        a = eval_acc(); acc_curve.append((step, a))
        print(f"step {step:4d}  loss {loss.item():.4f}  margin {(rc-rr).mean():+.3f}  acc {a:.4f}")
print("accuracy AFTER training:", round(eval_acc(), 4))

## 5 · Results

Plot the accuracy curve. On gpt2-small over the full HH mix you should reach **~0.57–0.60** — clearly above the 0.50 chance line. (Bigger models + more data push this higher; the point is that a *single scalar* learned purely from comparisons captures human preference.)

In [ ]:
# ✅ PROVIDED — plot the climb
import matplotlib.pyplot as plt
xs, ys = zip(*acc_curve)
plt.figure(figsize=(6,4))
plt.axhline(0.5, ls="--", c="gray", label="chance")
plt.plot(xs, ys, "-o", c="#8E6B2F", label="reward model")
plt.xlabel("training step"); plt.ylabel("preference accuracy"); plt.ylim(0.45, 0.7)
plt.title("The reward model learns human preference"); plt.legend(); plt.show()

## 6 · Questions to answer

1. The loss only uses the **difference** `r_chosen − r_rejected`. What does that say about the absolute reward values — and why will we need to **normalize** rewards before using them in PPO?
2. We left-truncated the input. Predict what happens to the final accuracy if you switch to `truncation_side="right"`. (Try it.)
3. Accuracy plateaus well below 100%. Name two reasons (think: model size, label noise in human preferences).
4. **Save your reward model** (`model.save_pretrained("rm")`) — you'll plug it into the flagship RLHF project as the reward signal.
